# 1. Conexion con SQL Server

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from sqlalchemy import create_engine
from sqlalchemy.types import Integer, String
import urllib

# 1. Configurar Faker y conexion a SQL
dates = Faker('es_CO') # Datos reales en español
acces = urllib.parse.quote_plus( # Codigo de acceso a SQL
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-RSEQD08\\SQLEXPRESS;"
    "DATABASE=techno;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={acces}") # Motor de acceso

# 2. Creacion de Tablas

In [3]:
# DimClientes (100 Clientes unicos)
# DimProductos (15 productos)
# DimTiendas (5 tiendas)
# factVentas (2000 registros)

np.random.seed(42) # Bloquea aleatoriedad de Numpy
dates.seed_instance(42) # Bloquea la aleatoriedad de dates

# Clientes
id_cliente = range(1, 101)
clientes = [dates.name() for _ in range(100)]
ciudad_cliente = [dates.city() for _ in range(100)]
edades = np.random.randint(18, 66, 100)
cedulas = [dates.random_number(digits=10, fix_len=True) for _ in range(100)]

dim_clientes = pd.DataFrame({
    'Id_Cliente': id_cliente, 
    'Nombre_Cliente': clientes,
    'Ciudad_Cliente': ciudad_cliente,
    'Edad_Cliente': edades,
    'Cedula_Cliente': cedulas
})

# Productos
id_Producto = range(1, 16) # Rango de 1 a 15
productos = ['Laptop','Mouse','Teclado','Monitor','Audifonos','Cargador','Silla',
            'Escritorio','Telefono','Tablet','Pc_Escritorio','Impresora','Proyector','Parlante','Dron']
precio_Unitario = np.round(np.random.uniform(1000000.00, 5000000.00, 15), 2)

dim_Productos = pd.DataFrame({
    'Id_Producto': id_Producto,
    'Desc_Producto': productos,
    'Precio_Unitario': precio_Unitario
})

# Tiendas
dim_Tiendas = pd.DataFrame({
    'Id_Tiendas': range(1, 6), # Rango de 1 a 5
    'Tiendas': ['GamerZone','SoundStore','VideoStar','ClickPower','PadStore'],
    'Zonas': ['Centro','Sur','Norte','Oriente','Occidente']
    
})

# Ventas
fact_Ventas = pd.DataFrame({
    'Id_Venta': range(1,2001),
    'Id_Cliente': np.random.choice(dim_clientes['Id_Cliente'],2000),
    'Id_Producto': np.random.choice(dim_Productos['Id_Producto'],2000),
    'Id_Tiendas': np.random.choice(dim_Tiendas['Id_Tiendas'],2000),
    'Cantidad': np.random.randint(1,6,2000),
    'Fecha': pd.to_datetime(np.random.choice(pd.date_range('2024-01-01', '2026-05-14'), 2000))
})

# 3. Enviar todo a SQL

In [ ]:
# Enviar todo a SQL Server
# 'replace' creará las tablas automáticamente en SQL Server si no existen
dim_clientes.to_sql('dim_clientes', con=engine, if_exists='replace', index=False,
dtype={
    "Id_Cliente": Integer(), 
    "Nombre_Cliente": String(100),
    "Ciudad_Cliente": String(100), 
    "Edad_Cliente": Integer()
})

dim_Productos.to_sql('dim_Productos', con=engine, if_exists='replace', index=False,
dtype= {
    'Id_Producto': Integer(),
    'Desc_Producto': String(100),
    'Precio_Unitario': Integer()
})

dim_Tiendas.to_sql('dim_Tiendas', con=engine, if_exists='replace', index=False,
dtype= {
    'Id_Tiendas': Integer(),
    'Tiendas': String(100),
    'Zonas': String(100)
})

fact_Ventas.to_sql('fact_Ventas', con=engine, if_exists='replace', index=False,
dtype={
    'Cantidad': Integer()
})